# Multi-Provider Agent Harness — Walkthrough

This notebook runs the **same execution engine** the worker runs, driven by the
**mock provider** so it works with **zero API keys and zero external infra**.
Flip `LIVE=True` (with keys + the Docker stack) to run against real
Claude/OpenAI/Gemini, real Postgres/S3, and a real MCP server.

Set a breakpoint anywhere in `harness/` (e.g. `harness/core/engine.py` inside
`_agent_loop`, or `harness/core/trace.py:Tracer.emit`) and run a cell to stop
with the full **neutral state** in scope — the loop never holds vendor objects.

In [ ]:
import sys, os, asyncio, json, tempfile
sys.path.insert(0, os.path.abspath('..'))   # repo root on path
os.environ.setdefault('ARTIFACTS_ROOT', tempfile.mkdtemp())
print('artifacts →', os.environ['ARTIFACTS_ROOT'])

## 1. Build an offline engine

One shared `MockProvider` serves every chain link (so the call counter is shared across providers and delegated sub-agents). The package's real provider names are aliased to it.

In [ ]:
from harness.core.engine import ExecutionEngine
from harness.core.factory import load_packages
from harness.core.trace import Tracer
from harness.providers.base import ModelChain
from harness.providers.mock_provider import MockProvider
from harness.tools.gateway import ToolGateway
from harness.connectors.base import ConnectorRegistry
from harness.connectors.binder import Binder
from harness.decisions.assembler import FileSink
from harness.hitl.continuation import FileContinuationStore, HumanDecision
from harness.core.result import HITLSuspended
from scripts.scenarios import SCENARIOS
from scripts.demo_app import build_demo_tools

packages = load_packages('../packages')

def offline_engine(turns, step=False):
    mock = MockProvider(turns=turns)
    providers = {n: mock for n in ('anthropic','openai','gemini','mock')}
    art = os.environ['ARTIFACTS_ROOT']
    return ExecutionEngine(
        chain=ModelChain(providers),
        tool_gateway=ToolGateway(python_tools=build_demo_tools()),
        binder=Binder(ConnectorRegistry()),
        packages=packages,
        sink=FileSink(art),
        continuation_store=FileContinuationStore(f'{art}/suspensions'),
        tracer=Tracer(enabled=True, step=step),
    )
print('packages:', list(packages))

## 2. Run a TASK (single-turn, forced structured output)

`classify_document` emits a structured classification; its S3 target is suppressed offline.

In [ ]:
name, kind, inp, turns, mock = SCENARIOS['classify']()
eng = offline_engine(turns)
res = await eng.run_task(task_name=name, input_data=inp, mock=mock)
res.output

## 3. Run the flagship AGENT

`underwriting_agent` exercises: a Postgres **source** (mocked), a python **tool**, an **MCP** tool (mocked), **delegation** to a sub-agent, and a **HITL gate** on `issue_binder`.

It will **suspend** at the gate — watch the trace nest the sub-agent at depth 1.

In [ ]:
name, kind, inp, turns, mock = SCENARIOS['underwriting']()
eng = offline_engine(turns)          # step=True to single-step the loop
suspension_id = None
try:
    res = await eng.run_agent(agent_name=name, context=inp, mock=mock)
    print('completed without suspension:', res.output)
except HITLSuspended as s:
    suspension_id = s.suspension_id
    print('SUSPENDED at gate', s.gate_tool, '→', suspension_id)

## 4. Human decision → resume

The suspended run is just a file. Record an approval and resume; the engine rebuilds the loop state, injects the decision as the gated tool's result, and continues to `submit_output`.

In [ ]:
await eng.continuations.record_decision(suspension_id, HumanDecision(decision='approve', decided_by='notebook'))
res = await eng.resume(suspension_id)
print('status:', res.status.value)
res.output

## 5. Inspect the decision log (ADR-0015 layout)

`runs/{yyyy}/{mm}/{dd}/{run_id}/decision_log.json` + `model_invocations.jsonl`.

In [ ]:
import glob
paths = sorted(glob.glob(os.environ['ARTIFACTS_ROOT'] + '/runs/**/decision_log.json', recursive=True))
print(len(paths), 'decision records (incl. the depth-1 sub-agent and the resumed tail)')
d = json.load(open(paths[-1]))
{k: d[k] for k in ['entity_name','entity_kind','decision_depth','status','tool_calls_made','source_resolutions','target_writes','models_used']}

## 6. Audit reproduction (deterministic replay)

Replay a recorded run with every gateway in playback mode — the model from its recording, tools/sources canned, targets suppressed. Cleanest on single-shot task recordings.

In [ ]:
from harness.mock.context import MockContext
sink = FileSink(os.environ['ARTIFACTS_ROOT'])
# find the classify task run (single clean recording)
target=None
for p in paths:
    dd=json.load(open(p))
    if dd['entity_name']=='classify_document' and not dd.get('reproduced_from_decision_id'):
        target=os.path.basename(os.path.dirname(p)); break
recorded = sink.load_model_invocations(target)
replay = MockProvider(replay=recorded)
eng2 = ExecutionEngine(chain=ModelChain({n:replay for n in ('anthropic','openai','gemini','mock')}),
                       tool_gateway=ToolGateway(python_tools=build_demo_tools()),
                       binder=Binder(ConnectorRegistry()), packages=packages, sink=sink, tracer=Tracer(enabled=False))
orig = json.load(open([p for p in paths if target in p][0]))
rep = await eng2.run_task(task_name='classify_document', input_data=orig['input_json'],
                          mock=MockContext(model_replay=recorded, mock_all_tools=True, suppress_targets=True))
print('original :', orig['output_json'])
print('reproduced:', rep.output)
assert orig['output_json']==rep.output, 'replay diverged!'
print('✓ identical')

## Going LIVE

With the Docker stack up and keys in `.env`:

```
docker compose --env-file .env up --build
```

then in this notebook (running in the `jupyter` service) the env already points
`PG_MAIN_DSN`, `S3_ENDPOINT_URL`, and `MCP_POLICY_URL` at the real services.
Build the engine with `harness.core.factory.build_default_engine(...)` instead of
`offline_engine`, and run with real inputs — the chain falls back
Claude → OpenAI → Gemini on exhausted retries.